# 12 · Decorators and Context Managers

After this notebook you can write decorators (with and without arguments) and context managers from scratch, explain exactly what `@` and `with` turn into, and build the `@tool` registry pattern that agent frameworks use to turn a Python function into a JSON schema a model can call.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [06 · Functions, args, scope and lambda](06_functions_args_scope_lambda.ipynb) (closures)

## Why this matters in interviews

- "Write a retry decorator with exponential backoff" and "write a timing decorator" are among the most common Python live-coding asks for AI and backend engineers.
- Every agent framework registers tools with a decorator — LangChain's `@tool`, the OpenAI Agents SDK's `@function_tool`, FastMCP's `@mcp.tool()`. Knowing what they do (signature + docstring → JSON schema) shows you understand function calling rather than just using it.
- Context managers are how Python guarantees cleanup: files, locks, DB sessions, tracing spans, temporarily patched settings. "What does `__exit__` returning `True` do?" is a favourite follow-up.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag04` | What makes a good tool definition? |
| `po08` | What is exponential backoff, and why is jitter not optional? |
| `po16` | How do you design a retry policy that does not make things worse? |
| `in25` | How do you handle rate limits properly? |

## 1. Functions are objects

A decorator is just a function that takes a function and returns a function. That only works because functions are ordinary objects: you can give them another name, pass them in, return them, and keep them in a dict.

**Predict:** does `yell("hello")` still work after `del shout`?

In [ ]:
import functools
import inspect
import json
import os
import random
import re
import time
from collections import Counter

def shout(text):
    """Return the text in upper case with a bang."""
    return text.upper() + "!"

yell = shout                          # a second name for the same object
del shout                             # removes a *name*, not the function
print(yell("hello"), "| __name__:", yell.__name__, "| __doc__:", yell.__doc__)

def make_prefixer(prefix):            # returns a new function that remembers `prefix` (a closure)
    def add_prefix(text):
        return f"{prefix}{text}"
    return add_prefix

as_user = make_prefixer("user: ")
formatters = {"upper": str.upper, "user": as_user}          # functions stored in a dict
print(formatters["user"]("where is my order?"), "|", formatters["upper"]("ok"))

## 2. From a wrapper to `@decorator`

A wrapper runs code before and after the original function. `@log_calls` above a `def` is **exactly** `add = log_calls(add)` after it — nothing more.

In [ ]:
def log_calls(fn):
    def wrapper(*args, **kwargs):
        print(f"  -> {fn.__name__}{args}")
        result = fn(*args, **kwargs)
        print(f"  <- {fn.__name__} returned {result!r}")
        return result                 # forgetting this line is the classic decorator bug
    return wrapper

def add(a, b):
    """Add two numbers."""
    return a + b

add = log_calls(add)                  # manual decoration...
add(2, 3)

@log_calls                            # ...and the same thing with the @ syntax
def multiply(a, b):
    """Multiply two numbers."""
    return a * b

multiply(4, 5)
print("__name__:", multiply.__name__, "| __doc__:", multiply.__doc__, "| signature:", inspect.signature(multiply))

The decorated function now **claims to be `wrapper`**: its name, docstring and signature are gone. That breaks `help()`, logs and tracebacks — and anything that reads the signature, such as a tool registry (section 12) or FastAPI. `functools.wraps` copies the metadata across and stores the original as `__wrapped__`, which `inspect.signature` follows:

In [ ]:
def log_calls(fn):
    @functools.wraps(fn)              # copy __name__, __doc__, __annotations__...; set __wrapped__
    def wrapper(*args, **kwargs):
        print(f"  -> {fn.__name__}{args}")
        return fn(*args, **kwargs)
    return wrapper

@log_calls
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

print("__name__:", multiply.__name__, "| __doc__:", multiply.__doc__, "| signature:", inspect.signature(multiply))
assert multiply.__name__ == "multiply" and multiply.__wrapped__(2, 3) == 6
assert str(inspect.signature(multiply)) == "(a: int, b: int) -> int"

## 3. Decorators with arguments

`@repeat(times=5)` has one more layer: `repeat(times=5)` is called first and **returns** the decorator. Three nested functions: the factory (takes the options), the decorator (takes the function), the wrapper (runs on every call).

As an AI-flavoured use: sample a noisy classifier several times and take a majority vote (the idea behind *self-consistency*).

In [ ]:
def repeat(times):                           # 1. factory: receives the decorator's arguments
    def decorator(fn):                       # 2. decorator: receives the function
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):        # 3. wrapper: runs on every call
            return [fn(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorator

rng = random.Random(0)

@repeat(times=5)                             # same as: noisy_label = repeat(times=5)(noisy_label)
def noisy_label(text):
    return rng.choice(["positive", "positive", "negative"])    # a classifier that is right 2/3 of the time

votes = noisy_label("great product, fast delivery")
print("votes:", votes, "-> majority:", Counter(votes).most_common(1)[0][0])
assert len(votes) == 5

## 4. Stacking order

```
@auth                       call:   auth -> cache -> log -> handler
@cache                      return: handler -> log -> cache -> auth
@log
def handler(): ...          same as: handler = auth(cache(log(handler)))
```

Decorators are **applied bottom-up** (the one nearest the `def` wraps first) but **run top-down** (the outermost wrapper sees the call first). It matters in practice: `@timed` above `@retry` measures the total time including every retry, while `@retry` above `@timed` times each attempt; a cache above a retry means a cache hit skips the retries entirely.

In [ ]:
trace = []

def tag(name):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            trace.append(f"{name}:before")
            result = fn(*args, **kwargs)
            trace.append(f"{name}:after")
            return result
        return wrapper
    return decorator

@tag("auth")
@tag("cache")
@tag("log")
def handler():
    trace.append("handler")
    return "ok"

handler()
print(" -> ".join(trace))
assert trace == ["auth:before", "cache:before", "log:before", "handler", "log:after", "cache:after", "auth:after"]

## 5. A timing decorator — usable with or without arguments

A common interview extension: make one decorator work both as `@timed` and as `@timed(label="...")`. The trick is a first parameter that is the function when used bare and `None` when called with options. Timings are kept on the wrapper so you can inspect them later; `try/finally` records the time even if the call raises.

In [ ]:
def timed(fn=None, *, label=None):
    if fn is None:                                   # used as @timed(label=...)
        return lambda f: timed(f, label=label)

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            wrapper.timings_ms.append((time.perf_counter() - start) * 1000)

    wrapper.timings_ms = []
    wrapper.label = label or fn.__name__
    return wrapper

@timed
def tokenize(text):
    return re.findall(r"\w+", text.lower())

@timed(label="fake retrieval")
def retrieve(query):
    time.sleep(0.02)                                 # pretend to wait on a vector DB
    return ["chunk-1", "chunk-2"]

for _ in range(3):
    tokenize("How long do refunds take? " * 200)
    retrieve("refunds")
for f in (tokenize, retrieve):
    print(f"{f.label:<15} calls={len(f.timings_ms)}  mean={sum(f.timings_ms) / len(f.timings_ms):6.2f} ms")
assert len(retrieve.timings_ms) == 3 and min(retrieve.timings_ms) >= 15

## 6. Retry with exponential backoff and jitter

The decorator every LLM client needs, done properly (po08, po16, in25):

1. **Retry only retryable errors** — the `retry_on` tuple. A `ValueError` from a bad request passes straight through.
2. **Exponential backoff with a cap** — the ceiling doubles each attempt (`base * 2**attempt`) but never exceeds `cap`.
3. **Full jitter** — wait a random time between 0 and the ceiling, so clients that failed together do not retry together.
4. **Caps on attempts and on total waiting**, so retries fit inside the caller's deadline.

`sleep` and `rng` are parameters so the demo (and your unit tests) can record the waits instead of sleeping.

In [ ]:
def retry(*, attempts=5, base=0.5, cap=8.0, retry_on=(TimeoutError, ConnectionError),
          jitter=True, max_total_wait=None, sleep=time.sleep, rng=None):
    rng = rng or random.Random()

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            waited = 0.0
            for attempt in range(attempts):
                try:
                    return fn(*args, **kwargs)
                except retry_on:
                    ceiling = min(cap, base * 2 ** attempt)
                    delay = rng.uniform(0, ceiling) if jitter else ceiling
                    out_of_budget = max_total_wait is not None and waited + delay > max_total_wait
                    if attempt == attempts - 1 or out_of_budget:
                        raise                                  # give up: re-raise the last error
                    waited += delay
                    sleep(delay)
        return wrapper
    return decorator

In [ ]:
waits, attempts_made = [], []
outcomes = iter([TimeoutError("read timed out"), ConnectionError("connection reset"), TimeoutError("read timed out")])

@retry(attempts=5, base=0.5, sleep=waits.append, rng=random.Random(0))
def flaky_llm_call(prompt):
    attempts_made.append(prompt)
    if (err := next(outcomes, None)) is not None:
        raise err
    return f"answer to {prompt!r}"

print(flaky_llm_call("refund policy?"), "| attempts:", len(attempts_made), "| waits:", [round(w, 2) for w in waits])

bad_waits = []

@retry(attempts=5, sleep=bad_waits.append)
def bad_request():
    raise ValueError("400: 'temperature' must be <= 2")

try:
    bad_request()
except ValueError as e:
    print("not retried:", e, "| waits:", bad_waits)
assert len(attempts_made) == 4 and len(waits) == 3 and bad_waits == []
assert all(0 <= w <= min(8.0, 0.5 * 2 ** i) for i, w in enumerate(waits))     # each wait is under its ceiling

What does the schedule look like? The line is the ceiling without jitter (0.5, 1, 2, 4, then capped at 8 s); the dots are 200 simulated clients using full jitter — each waits anywhere from 0 up to the ceiling.

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True})

base, cap, n_attempts = 0.5, 8.0, 7
ceilings = [min(cap, base * 2 ** a) for a in range(n_attempts)]
sim = random.Random(1)
fig, ax = plt.subplots(figsize=(9, 3.8))
for a, ceiling in enumerate(ceilings):
    xs = [a + 1 + sim.uniform(-0.18, 0.18) for _ in range(200)]
    ys = [sim.uniform(0, ceiling) for _ in range(200)]
    ax.scatter(xs, ys, s=8, color=BLUE, alpha=0.35, label="full jitter: uniform(0, ceiling)" if a == 0 else None)
ax.plot(range(1, n_attempts + 1), ceilings, color=ORANGE, lw=2, marker="o", label="no jitter: min(cap, base * 2^attempt)")
ax.set_xlabel("retry number"); ax.set_ylabel("wait before this retry (s)")
ax.set_title("Exponential backoff with a cap of 8 s; jitter spreads each wait between 0 and the ceiling")
ax.legend(frameon=False, loc="upper left")
plt.show()

Why jitter is **not optional**: imagine 1,000 clients all hit a 429 at the same moment, each using a typical policy (base 1 s, cap 8 s, 4 retries). Without jitter they all retry at exactly 1 s, 3 s, 7 s and 15 s — four synchronised waves that knock the recovering service over again. With full jitter the same retries are spread out.

In [ ]:
def retry_times(n_clients, jitter, base=1.0, cap=8.0, retries=4, seed=0):
    r = random.Random(seed)
    times = []
    for _ in range(n_clients):
        t = 0.0
        for a in range(retries):
            ceiling = min(cap, base * 2 ** a)
            t += r.uniform(0, ceiling) if jitter else ceiling
            times.append(t)
    return times

bins = [i * 0.1 for i in range(0, 161)]
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True, sharey=True)   # same scale: honest
counts_plain, _, _ = top.hist(retry_times(1000, jitter=False), bins=bins, color=ORANGE)
counts_jitter, _, _ = bottom.hist(retry_times(1000, jitter=True), bins=bins, color=BLUE)
top.set_title("1,000 clients fail together at t=0: when do their retries land?")
top.set_ylabel("no jitter\nretries / 0.1 s"); bottom.set_ylabel("full jitter\nretries / 0.1 s")
bottom.set_xlabel("seconds after the shared failure")
plt.show()
print(f"busiest 0.1 s window: {counts_plain.max():.0f} retries without jitter vs {counts_jitter.max():.0f} with full jitter")
assert counts_plain.max() == 1000 and counts_jitter.max() < 300

Production additions, covered in [retries, backoff, rate limits and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb): honour a `Retry-After` header when the provider sends one, add an idempotency key to anything with side effects, and cap retries fleet-wide so an outage cannot triple your traffic.

## 7. Memoization with `lru_cache`

`functools.lru_cache` is a decorator that remembers results by argument. Two classic demos: recursion that recomputes the same subproblems, and an expensive call made repeatedly with the same input.

In [ ]:
plain_calls = 0

def fib_plain(n):
    global plain_calls
    plain_calls += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

assert fib_plain(25) == fib(25) == 75025
print(f"fib(25): {plain_calls:,} calls without a cache vs {fib.cache_info().misses} with one")
print("fib(300) is now instant:", str(fib(300))[:20] + "...")

@functools.lru_cache(maxsize=1024)
def embed(text):
    time.sleep(0.01)                          # pretend this is a paid API call
    return [len(text), text.count(" ")]

for q in ["refund policy", "shipping time", "refund policy", "refund policy"]:
    embed(q)
print(embed.cache_info())
try:
    embed(["a", "list"])                      # arguments become dict keys, so they must be hashable
except TypeError as e:
    print("TypeError:", e)
assert embed.cache_info().hits == 2

Rules of thumb:

- cache only **pure** functions (same input, same output, no side effects) — never a call whose answer should change, like "current stock level";
- arguments must be **hashable**: pass tuples, not lists;
- the cache returns the **same object** each time — mutate it and you corrupt the cache (the `embed` result above is a list: don't `.append` to it);
- `lru_cache` on a method keeps every `self` alive; prefer `functools.cached_property` for per-object values;
- for LLM calls the key must include the model, the parameters and the full prompt — exact and semantic caches for LLMs are in [caching, exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb).

## 8. Class-based decorators

When a decorator needs **state** — a counter, a rate limiter, a circuit breaker — a class is often clearer: `__init__` receives the function, `__call__` runs on every call, and the state lives on the instance. `functools.update_wrapper` does what `@wraps` does.

In [ ]:
class CountCalls:
    def __init__(self, fn):
        functools.update_wrapper(self, fn)
        self.fn = fn
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.fn(*args, **kwargs)

@CountCalls
def moderate(text):
    """Flag text containing a banned word."""
    return any(word in text.lower() for word in ("scam", "hack"))

results = [moderate(t) for t in ["hello", "is this a scam?", "hi"]]
print("results:", results, "| calls:", moderate.count, "| name:", moderate.__name__, "| doc:", moderate.__doc__)
assert moderate.count == 3 and results == [False, True, False]

Trap: an instance of a class like this does not bind `self` when used on a **method** (it has no `__get__`), so `@CountCalls` on a method breaks. Function-based decorators do not have this problem.

## 9. Context managers: what `with` really does

`with EXPR as x: BODY` is roughly:

```python
manager = EXPR
x = manager.__enter__()
try:
    BODY
except BaseException as e:
    if not manager.__exit__(type(e), e, e.__traceback__):   # True means "handled, stop here"
        raise
else:
    manager.__exit__(None, None, None)
```

So `__exit__` **always** runs, and it is told about any exception. Returning `True` swallows it; returning `False` or `None` lets it propagate.

In [ ]:
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self                                   # this is what `as t` receives

    def __exit__(self, exc_type, exc, tb):
        self.elapsed_ms = (time.perf_counter() - self.start) * 1000
        return False                                  # never swallow errors

class Suppress:
    """A tiny version of contextlib.suppress that remembers what it caught."""
    def __init__(self, *types):
        self.types, self.caught = types, None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None and issubclass(exc_type, self.types):
            self.caught = exc
            return True                               # handled: the exception stops here
        return False                                  # anything else keeps propagating

with Timer() as t:
    sum(i * i for i in range(200_000))
print(f"Timer: {t.elapsed_ms:.1f} ms")

with Suppress(KeyError) as s:
    {}["missing"]
print("Suppress swallowed:", repr(s.caught))
try:
    with Suppress(KeyError):
        1 / 0
except ZeroDivisionError:
    print("ZeroDivisionError passed straight through")
assert isinstance(s.caught, KeyError)

## 10. `contextlib.contextmanager` — the short way

Write a generator: everything before `yield` is `__enter__`, everything after is `__exit__`, and the yielded value is what `as` receives. Put the `yield` inside `try/finally`, or your cleanup will be skipped when the body raises.

Two everyday examples — a timer block, and a manager that **temporarily** sets environment variables and always restores them (handy for tests and for switching providers, e.g. `LLM_PROVIDER`):

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"[{label}] {(time.perf_counter() - start) * 1000:.0f} ms")

@contextmanager
def temp_env(**overrides):
    """Set environment variables for the duration of the block, then restore the old values."""
    saved = {key: os.environ.get(key) for key in overrides}
    os.environ.update({key: str(value) for key, value in overrides.items()})
    try:
        yield
    finally:
        for key, old in saved.items():
            if old is None:
                os.environ.pop(key, None)            # it did not exist before: remove it
            else:
                os.environ[key] = old

with timer("fake retrieval"):
    time.sleep(0.05)

os.environ["NB12_MODEL"] = "gpt-4.1-mini"
try:
    with temp_env(NB12_MODEL="gpt-4.1-nano", NB12_DEBUG=1):
        print("inside :", os.environ["NB12_MODEL"], os.environ["NB12_DEBUG"])
        raise RuntimeError("boom")                    # cleanup must still happen
except RuntimeError:
    pass
print("after  :", os.environ["NB12_MODEL"], "| NB12_DEBUG still set?", "NB12_DEBUG" in os.environ)
assert os.environ["NB12_MODEL"] == "gpt-4.1-mini" and "NB12_DEBUG" not in os.environ

In tests, `unittest.mock.patch.dict(os.environ, {...})` does the same job ([15 · Testing and debugging](15_testing_and_debugging.ipynb)).

## 11. `suppress` and `ExitStack`

- `contextlib.suppress(FileNotFoundError)` — ignore one specific, expected error, readably. (For this particular case `Path.unlink(missing_ok=True)` also exists.)
- `contextlib.ExitStack` — enter a **variable number** of context managers (open one file per data shard, one span per retrieved doc) and have them all closed at the end, in reverse order. `stack.callback(fn)` registers any cleanup function.

In [ ]:
import shutil
import tempfile
from contextlib import ExitStack, suppress
from pathlib import Path

shard_dir = Path(tempfile.mkdtemp(prefix="nb12_"))
with suppress(FileNotFoundError):
    (shard_dir / "does_not_exist.txt").unlink()
print("suppress: no error raised")

splits = ["train", "valid", "test"]
with ExitStack() as stack:
    stack.callback(shutil.rmtree, shard_dir)                     # registered first, so it runs LAST
    stack.callback(lambda: print("line counts:", {s: (shard_dir / f"{s}.jsonl").read_text(encoding="utf-8").count("\n")
                                                 for s in splits}))
    files = {s: stack.enter_context(open(shard_dir / f"{s}.jsonl", "w", encoding="utf-8")) for s in splits}
    for i in range(10):
        split = "train" if i < 8 else ("valid" if i == 8 else "test")
        files[split].write(json.dumps({"id": i}) + "\n")
print("all closed:", all(f.closed for f in files.values()), "| folder removed:", not shard_dir.exists())
assert all(f.closed for f in files.values()) and not shard_dir.exists()

Context managers you already use:

| Manager | Guarantees on exit |
|---|---|
| `open(path)` | the file is closed |
| `threading.Lock()` | the lock is released |
| `tempfile.TemporaryDirectory()` | the folder is deleted |
| `unittest.mock.patch(...)` | the original object is restored |
| `contextlib.redirect_stdout(buffer)` | `sys.stdout` is restored |
| `contextlib.chdir(path)` (3.11+) | the working directory is restored |
| `async with asyncio.timeout(5)` | the block is cancelled if it runs too long ([notebook 14](14_async_await_basics.ipynb)) |

## 12. Practical: a `@tool` registry that builds OpenAI-style JSON schemas

This is what agent frameworks do when you decorate a function as a tool. Using `inspect.signature` (parameter names and defaults), `typing.get_type_hints` (types) and the docstring (descriptions), we build the function-calling schema the model receives:

```
def get_weather(city: str, unit: Literal[...] = "celsius")   ->   {"name": "get_weather",
    """Get the current weather...                                  "description": "...",
    Args:                                                          "parameters": {"type": "object",
        city: City name, e.g. "Paris".                                "properties": {...},
    """                                                               "required": ["city"]}}
```

The quality rules from `ag04` fall out naturally: a verb_noun **name**, a **description** that says when *not* to use the tool, **enums** (`Literal`) instead of free strings, required vs optional taken from the defaults, and errors that tell the model how to fix its call.

In [ ]:
import types
import typing
from typing import Literal, get_args, get_origin

PY_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean", dict: "object"}

def json_schema_for(tp):
    """Map a Python type hint to a JSON-schema fragment."""
    origin = get_origin(tp)
    if origin is Literal:
        values = list(get_args(tp))
        return {"type": PY_TO_JSON[type(values[0])], "enum": values}
    if origin in (types.UnionType, typing.Union):           # X | None  or  Optional[X]
        non_null = [a for a in get_args(tp) if a is not type(None)]
        if len(non_null) == 1:
            return json_schema_for(non_null[0])
    if origin is list:
        return {"type": "array", "items": json_schema_for(get_args(tp)[0])}
    if tp in PY_TO_JSON:
        return {"type": PY_TO_JSON[tp]}
    raise TypeError(f"unsupported parameter type: {tp!r}")

def parse_docstring(doc):
    """Google-style docstring -> (description, {param: description})."""
    description, params, section = [], {}, "description"
    for line in (doc or "").splitlines():
        stripped = line.strip()
        if stripped in ("Args:", "Returns:", "Raises:"):
            section = stripped
        elif section == "description" and stripped:
            description.append(stripped)
        elif section == "Args:" and (m := re.match(r"(\w+)\s*:\s*(.+)", stripped)):
            params[m[1]] = m[2]
    return " ".join(description), params

In [ ]:
TOOLS = {}

def tool(fn=None, *, name=None):
    """Register a function as a model-callable tool and attach its JSON schema as fn.tool_spec."""
    def register(f):
        description, param_docs = parse_docstring(inspect.getdoc(f))
        hints = typing.get_type_hints(f)
        properties, required = {}, []
        for pname, param in inspect.signature(f).parameters.items():
            schema = json_schema_for(hints.get(pname, str))
            if pname in param_docs:
                schema["description"] = param_docs[pname]
            if param.default is inspect.Parameter.empty:
                required.append(pname)
            else:
                schema["default"] = param.default
            properties[pname] = schema
        spec = {"type": "function",
                "function": {"name": name or f.__name__, "description": description,
                             "parameters": {"type": "object", "properties": properties,
                                            "required": required, "additionalProperties": False}}}
        TOOLS[spec["function"]["name"]] = {"fn": f, "spec": spec}
        f.tool_spec = spec
        return f                          # the original function: still callable as normal Python
    return register(fn) if fn is not None else register

In [ ]:
@tool
def get_weather(city: str, unit: Literal["celsius", "fahrenheit"] = "celsius") -> str:
    """Get the current weather for one city.

    Args:
        city: City name, e.g. "Paris".
        unit: Temperature unit for the answer.
    """
    return f"18 degrees {unit}, cloudy, in {city}"

@tool(name="search_orders")
def find_orders(email: str, status: Literal["OPEN", "PENDING", "REFUNDED", "CLOSED"] | None = None,
                limit: int = 20) -> list[dict]:
    """Find orders for ONE customer by email, newest first.

    Do NOT use this to count orders; use count_orders for that.

    Args:
        email: The customer's email address.
        status: Only return orders with this status.
        limit: Maximum number of orders to return.
    """
    return [{"order_id": "ORD-123456", "status": status or "OPEN"}][:limit]

print(json.dumps(TOOLS["search_orders"]["spec"], indent=2))

In [ ]:
expected_weather = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for one city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": 'City name, e.g. "Paris".'},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                         "description": "Temperature unit for the answer.", "default": "celsius"},
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    },
}
assert get_weather.tool_spec == expected_weather
assert TOOLS["search_orders"]["spec"]["function"]["parameters"]["required"] == ["email"]
assert get_weather("Paris") == "18 degrees celsius, cloudy, in Paris"     # still a normal function
print("registered tools:", sorted(TOOLS), "- schemas match the expected JSON")

The other half of the loop: the model answers with a tool **call** — a name plus a JSON string of arguments. Dispatch it through the registry, validate the arguments against the same schema, and turn every problem into a message the model can act on (`ag04`: "status must be one of ...; you sent 'open'").

In [ ]:
JSON_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool, "object": dict, "array": list}

def validate_args(args, parameters):
    problems = [f"missing required argument {r!r}" for r in parameters["required"] if r not in args]
    for key, value in args.items():
        schema = parameters["properties"].get(key)
        if schema is None:
            problems.append(f"unknown argument {key!r}; expected one of {sorted(parameters['properties'])}")
        elif "enum" in schema and value not in schema["enum"]:
            problems.append(f"{key} must be one of {schema['enum']}; you sent {value!r}")
        elif not isinstance(value, JSON_TYPES[schema["type"]]):
            problems.append(f"{key} must be of type {schema['type']}; you sent {value!r}")
    return problems

def dispatch(tool_call):
    name, raw_args = tool_call["function"]["name"], tool_call["function"]["arguments"]
    if name not in TOOLS:
        return json.dumps({"error": f"unknown tool {name!r}; available: {sorted(TOOLS)}"})
    args = json.loads(raw_args)
    if problems := validate_args(args, TOOLS[name]["spec"]["function"]["parameters"]):
        return json.dumps({"error": "; ".join(problems)})
    return json.dumps({"result": TOOLS[name]["fn"](**args)})

calls = [("get_weather", '{"city": "Paris"}'),
         ("search_orders", '{"email": "a@b.com", "status": "open"}'),
         ("search_orders", '{"status": "OPEN", "limit": "5"}'),
         ("get_stock_price", '{"ticker": "ACME"}')]
for name, arguments in calls:
    call = {"id": "call_1", "type": "function", "function": {"name": name, "arguments": arguments}}
    print(f"{name}({arguments})\n   -> {dispatch(call)}")
assert "must be one of" in dispatch({"function": {"name": "search_orders", "arguments": '{"email": "x", "status": "open"}'}})

In a real agent the `spec` dictionaries go into `client.chat.completions.create(..., tools=[...])` and the calls come back in `response.choices[0].message.tool_calls` — that loop is in [function calling](../08_llm_apps/04_function_calling.ipynb) and [the agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb). Frameworks do the type-to-schema step with pydantic, which also handles nested objects and constraints; OpenAI's `"strict": true` mode additionally requires every property to be listed in `required` — [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb) shows the SDK helper that produces such a schema.

## Try it yourself

**1.** Write a function-based `@count_calls` decorator that stores the number of calls on the wrapper as `wrapper.calls`, and keeps the function's name.

In [ ]:
# Solution — try it yourself first, then run this
def count_calls(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def chunk(text, size=10):
    return [text[i:i + size] for i in range(0, len(text), size)]

chunk("a" * 25); chunk("b" * 5)
print(chunk.__name__, "was called", chunk.calls, "times")
assert chunk.calls == 2 and chunk.__name__ == "chunk"

**2.** Write a `@log_tool_io` decorator that prints a tool's arguments and result. Stack it **under** `@tool` and check that the generated schema is identical to the undecorated version — this only works because of `functools.wraps`.

In [ ]:
# Solution — try it yourself first, then run this
def log_tool_io(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"[tool] {fn.__name__} args={kwargs or args} -> {result!r}")
        return result
    return wrapper

@tool(name="get_weather_logged")
@log_tool_io
def get_weather_logged(city: str, unit: Literal["celsius", "fahrenheit"] = "celsius") -> str:
    """Get the current weather for one city.

    Args:
        city: City name, e.g. "Paris".
        unit: Temperature unit for the answer.
    """
    return f"18 degrees {unit}, cloudy, in {city}"

get_weather_logged(city="Oslo")
same = get_weather_logged.tool_spec["function"]["parameters"] == get_weather.tool_spec["function"]["parameters"]
print("schema unchanged by the extra decorator:", same)
assert same

**3.** Write a `span(name)` context manager with `@contextmanager` that appends `(name, milliseconds)` to a list when the block ends — a baby tracing span. Nest a `"retrieve"` and a `"generate"` span inside a `"request"` span and check the order they finish in.

In [ ]:
# Solution — try it yourself first, then run this
spans = []

@contextmanager
def span(name):
    start = time.perf_counter()
    try:
        yield
    finally:
        spans.append((name, round((time.perf_counter() - start) * 1000, 1)))

with span("request"):
    with span("retrieve"):
        time.sleep(0.01)
    with span("generate"):
        time.sleep(0.02)
print(spans)
assert [name for name, _ in spans] == ["retrieve", "generate", "request"]   # inner spans finish first
assert spans[-1][1] >= spans[0][1] + spans[1][1] - 1                     # the parent contains its children

## Say it in an interview

- **Decorator in one sentence:** "A function that takes a function and returns a replacement; `@d` above a `def` is `f = d(f)`. With arguments it is a factory: `@d(x)` is `f = d(x)(f)`. I always use `functools.wraps` so the name, docstring and signature survive — frameworks that read signatures depend on it."
- **Stacking:** "Applied bottom-up, executed top-down. Order matters: timer outside retry measures total latency; cache outside retry skips retries on a hit."
- **Retry decorator (po08 / po16 / in25):** "Classify first — only transient errors are retried. Exponential backoff with a cap and full jitter, a maximum number of attempts and a total time budget, `Retry-After` honoured, idempotency keys for side effects. Jitter matters because synchronised retries re-break a recovering service."
- **lru_cache:** "Pure functions with hashable arguments; `cache_info()` shows hits and misses; it returns the same object, so don't mutate results; on methods it keeps instances alive."
- **Context managers:** "`__enter__`/`__exit__`; exit always runs and receives the exception; returning `True` suppresses it. `@contextmanager` needs `try/finally` around the `yield`. `ExitStack` handles a dynamic number of resources."
- **Tool registry (ag04):** "Signature → required/optional and defaults, type hints → JSON types and enums, docstring → descriptions. Good tools have verb_noun names, say when not to use them, use enums, and return errors that tell the model how to fix the call."
- **Traps:** forgetting `return` in the wrapper; forgetting `@wraps`; a class-based decorator on a method; `yield` in a context manager without `try/finally`; an `__exit__` that accidentally returns a truthy value and hides errors.

## Next

- [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb) — the same schemas, generated and validated by pydantic
- [14 · async/await basics](14_async_await_basics.ipynb) — `async with` and async decorators
- [15 · Testing and debugging](15_testing_and_debugging.ipynb) — `mock.patch` is a decorator *and* a context manager
- Agents: [tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb) · [function calling](../08_llm_apps/04_function_calling.ipynb) · [MCP](../15_agentic_ai/07_mcp_model_context_protocol.ipynb)
- Production: [retries, backoff, rate limits and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [caching, exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [agentic_ai_flow course](../../agentic_ai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)